# 01 国会会議録 API の返事の形を確かめる(下調べ)

**問い**: [国会会議録検索システム](https://kokkai.ndl.go.jp/) の API は、発言1件ごとにどんな項目を返すか。党を決めるのに使える項目はどれか。いつの発言まで登録されているか。

**方法**: API に少しだけ問い合わせて(1件・100件)、返ってきた項目と値を見る。本番の取得は `scripts/01_fetch_kokkai.py` で行う。

**結論(先に)**
- 1件ごとに、発言者・会派(`speakerGroup`)・役職(`speakerPosition`)・院・会議名・日付・本文などが返る
- 発言の順番(`speechOrder`)が 0 の行は発言ではなく、会議録の見出し(出席者の一覧など)→ 除く
- **会派の名前は、同じ議員でも時期によって変わる** → 会派の欄をそのまま党として使わず、会派の名前から党を決める対応表を作る(`data/external/group_party_map.csv`。02 で点検)
- 大臣など政府側の発言にも会派が付く。役職の欄で、議員としての発言と区別できる。委員長の司会は、発言の中身で別に除く
- 会議録は、国会が開かれてから**しばらく後に登録される**(下の 4.)→ 取得は 2025-01〜2026-07 にした

**流れ**: 準備 → 1. 返事の形 → 2. 会派の値 → 3. 役職と発言の順番 → 4. どの月まで登録されているか

※ 2026-10-25 に上から順に実行し直した。セルの実行番号が 4 から始まるのは、同じカーネルで先に試し実行したためで、順番は上から順になっている。

## 準備

使う道具(`requests` = Web に問い合わせる、`pandas` = 表を扱う)を読み込む。

In [4]:
import json, time
from pathlib import Path
import requests
import pandas as pd

- `fetch(**params)`: API に問い合わせて、返事(JSON)を辞書で返す。1回ごとに2秒待つ(サーバーに負担をかけないため)。`maximumRecords` は1回に受け取る件数
- `HEADERS`: 問い合わせるときに「誰が、何のために」を名乗る。名乗らない Python からの問い合わせは、2026-10 に **403(お断り)** で返された(ブラウザからは見られた)。名乗るようにしたら返事が来た

In [5]:
URL = "https://kokkai.ndl.go.jp/api/speech"
# 何のための問い合わせかを名乗る(名乗らない Python からの問い合わせは、2026-10 時点で 403 で断られた)
# 名乗りは src/collect/kokkai.py の USER_AGENT と同じ
HEADERS = {"User-Agent": "voter-party-comovement-japan-2025-2026/0.1 (+https://github.com/Taiyo-Tanabe/voter-party-comovement-japan-2025-2026)"}
RAW = Path("../data/raw/kokkai")
RAW.mkdir(parents=True, exist_ok=True)

def fetch(**params):
    base = {"recordPacking": "json", "maximumRecords": 1}
    r = requests.get(URL, params={**base, **params}, headers=HEADERS, timeout=30)
    print(r.url)
    r.raise_for_status()
    data = r.json()
    if "speechRecord" not in data:
        print(json.dumps(data, ensure_ascii=False, indent=2)[:1000])
    time.sleep(2)
    return data

## 1. 返事の形

2025年11月の発言を1件だけ受け取り、まず件数などの情報を見る。

In [6]:
data = fetch(**{"from": "2025-11-01", "until": "2025-11-30"})
print({k: v for k, v in data.items() if k != "speechRecord"})

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2025-11-01&until=2025-11-30
{'numberOfRecords': 11341, 'numberOfReturn': 1, 'startRecord': 1, 'nextRecordPosition': 2}


**結果**: 2025年11月の発言は **11,341 件**。次に、受け取った1件の中身(項目と値)を見る。

In [7]:
rec = data["speechRecord"][0]
for k, v in rec.items():
    print(f"{k:20} {str(v)[:80]!r}")

speechID             '121903968X00220251128_000'
issueID              '121903968X00220251128'
imageKind            '会議録'
searchObject         '0'
session              '219'
nameOfHouse          '衆議院'
nameOfMeeting        '外務委員会'
issue                '第2号'
date                 '2025-11-28'
closing              'None'
speechOrder          '0'
speaker              '会議録情報'
speakerYomi          'None'
speakerGroup         'None'
speakerPosition      'None'
speakerRole          'None'
speech               '令和七年十一月二十八日（金曜日）\r\n\u3000\u3000\u3000\u3000午前九時開議\r\n\u3000出席委員\r\n\u3000\u3000\u3000委員長\u3000國場幸之助君\r\n\u3000\u3000\u3000理事\u3000石橋林太郎君\u3000理事\u3000高木\u3000\u3000啓君\r\n\u3000\u3000\u3000理'
startPage            '1'
speechURL            'https://kokkai.ndl.go.jp/txt/121903968X00220251128/0'
meetingURL           'https://kokkai.ndl.go.jp/txt/121903968X00220251128'
pdfURL               'https://kokkai.ndl.go.jp/img/121903968X00220251128/1'


**結果**: 項目は、発言ID・会議ID・院(`nameOfHouse`)・会議名・日付・発言の順番(`speechOrder`)・発言者(`speaker`)・会派(`speakerGroup`)・役職(`speakerPosition`)・本文(`speech`)・URL など。

この1件目は、発言者が「会議録情報」、発言の順番が 0 で、本文は日付と出席委員の一覧だった → **発言ではなく、会議録の見出し**。

## 2. 会派(speakerGroup)の値

100件受け取り、会派の値を数える。院ごとにも分ける。

In [8]:
sample = fetch(
    **{
        "from": "2025-11-01", 
        "until": "2025-11-30", 
        "maximumRecords": 100
    }
)
df = pd.DataFrame(sample["speechRecord"])
df["speakerGroup"].value_counts(dropna=False)

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=100&from=2025-11-01&until=2025-11-30


speakerGroup
自由民主党・無所属の会    49
立憲民主党・無所属      43
NaN             6
自由民主党           2
Name: count, dtype: int64

In [9]:
pd.crosstab(df["speakerGroup"].fillna("(空)"), df["nameOfHouse"])

nameOfHouse,衆議院
speakerGroup,
(空),6
立憲民主党・無所属,43
自由民主党,2
自由民主党・無所属の会,49


**結果**: 100件はすべて衆議院(この月の最初の会議の1つ、外務委員会の発言)。会派は「自由民主党・無所属の会」49、「立憲民主党・無所属」43、空 6、「自由民主党」2。

同じ自民党なのに、会派の名前が2種類ある。「自由民主党」の2件が誰の発言かを見る。

In [10]:
cols = ["date", "nameOfMeeting", "speaker", "speakerPosition", "speakerRole", "speakerGroup"]
df[df["speakerGroup"] == "自由民主党"][cols]

,date,nameOfMeeting,speaker,speakerPosition,speakerRole,speakerGroup
15,2025-11-28,外務委員会,永井学,国土交通大臣政務官,None,自由民主党
19,2025-11-28,外務委員会,永井学,国土交通大臣政務官,None,自由民主党


**結果**: 「自由民主党」の2件は、永井学議員(国土交通大臣政務官)の答弁。次のセルで「自由民主党・無所属の会」の発言者を見る。

In [11]:
df[df["speakerGroup"] == "自由民主党・無所属の会"][cols].drop_duplicates("speaker").head()

,date,nameOfMeeting,speaker,speakerPosition,speakerRole,speakerGroup
1,2025-11-28,外務委員会,國場幸之助,NaN,None,自由民主党・無所属の会
4,2025-11-28,外務委員会,石橋林太郎,NaN,None,自由民主党・無所属の会
5,2025-11-28,外務委員会,茂木敏充,外務大臣,None,自由民主党・無所属の会
11,2025-11-28,外務委員会,島田智明,外務大臣政務官,None,自由民主党・無所属の会
74,2025-11-28,外務委員会,井野俊郎,経済産業副大臣・内閣府副大臣,None,自由民主党・無所属の会


**結果**: 委員長・外務大臣・政務官・副大臣も「自由民主党・無所属の会」。→ 永井議員だけ会派の名前が違う理由を、永井議員の発言全体(100件)で調べる。

In [12]:
d = fetch(**{"speaker": "永井学", "maximumRecords": 100})
pd.DataFrame(d["speechRecord"])[["nameOfHouse", "speakerGroup"]].value_counts()

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=100&speaker=%E6%B0%B8%E4%BA%95%E5%AD%A6


nameOfHouse  speakerGroup
参議院          自由民主党           74
             自由民主党・無所属の会     15
衆議院          自由民主党・無所属の会      7
             自由民主党            4
Name: count, dtype: int64

**結果**: 永井議員の発言 100件のうち、参議院で「自由民主党」74件・「自由民主党・無所属の会」15件、衆議院で「自由民主党・無所属の会」7件・「自由民主党」4件。時期ごとに分けて見る。

In [13]:
nd = pd.DataFrame(d["speechRecord"])
nd.groupby(["speakerGroup", "nameOfHouse"])["date"].agg(["min", "max", "count"])

min         max  count
speakerGroup nameOfHouse                               
自由民主党        参議院          2023-11-09  2025-11-20     74
             衆議院          2025-11-19  2025-11-28      4
自由民主党・無所属の会  参議院          2026-03-19  2026-05-28     15
             衆議院          2026-03-06  2026-06-03      7

**結果**: 2025-11 までは両院で「自由民主党」、2026-03 以降は両院で「自由民主党・無所属の会」。**同じ議員でも、時期によって会派の名前が変わる**。

→ 会派の名前を、そのまま党の名前として使うことはできない。会派の名前から党を決める対応表を作る(`group_party_map.csv`)。

## 3. 役職(speakerPosition)と、発言の順番 0

100件で、役職の値・`speakerRole` の値・発言の順番が 0 の本文を見る。

In [14]:
print(df["speakerPosition"].value_counts(dropna=False).head(10))
print(df["speakerRole"].value_counts(dropna=False))
df.loc[df["speechOrder"] == 0, "speech"].str[:200].tolist()

speakerPosition
NaN                        60
外務大臣                       28
国土交通大臣政務官                   2
厚生労働省大臣官房危機管理・医務技術総括審議官     2
経済産業副大臣・内閣府副大臣              2
内閣官房副長官                     2
外務省大臣官房審議官                  1
外務大臣政務官                     1
厚生労働省大臣官房審議官                1
公正取引委員会委員長                  1
Name: count, dtype: int64
speakerRole
None    100
Name: count, dtype: int64


['令和七年十一月二十八日（金曜日）\r\n\u3000\u3000\u3000\u3000午前九時開議\r\n\u3000出席委員\r\n\u3000\u3000\u3000委員長\u3000國場幸之助君\r\n\u3000\u3000\u3000理事\u3000石橋林太郎君\u3000理事\u3000高木\u3000\u3000啓君\r\n\u3000\u3000\u3000理事\u3000星野\u3000剛士君\u3000理事\u3000亀井亜紀子君\r\n\u3000\u3000\u3000理事\u3000末松\u3000義規君\u3000理事\u3000太\u3000\u3000栄志君\r\n\u3000\u3000\u3000理事\u3000杉本\u3000和巳君\u3000理事\u3000深作ヘスス君\r\n\u3000\u3000\u3000\u3000\u3000\u3000逢沢\u3000一郎君\u3000\u3000\u3000\u3000岩屋\u3000\u3000毅君\r\n\u3000\u3000\u3000\u3000\u3000\u3000英利アルフィヤ君\u3000\u3000\u3000\u3000大空\u3000幸星君\r\n\u3000\u3000']

**結果**
- 役職が空(= 議員としての発言)は 60件。外務大臣 28件、政務官・副大臣・官僚(審議官)などもいる → **役職の欄で、政府側の答弁と議員の質問を分けられる**
- `speakerRole` はすべて空だった(使えない)
- 発言の順番が 0 の本文は、日付と出席委員の一覧 → 発言ではないので除く

次に、役職が空で発言の順番が 1 以上の発言(議員としての発言)の会派と中身を見る。

In [15]:
q = df[df["speakerPosition"].isna() & (df["speechOrder"] > 0)]
print(q["speakerGroup"].value_counts(dropna=False))
q[["speaker", "speakerGroup", "speech"]].assign(speech=q["speech"].str[:40]).head(15)

speakerGroup
立憲民主党・無所属      43
自由民主党・無所属の会    16
Name: count, dtype: int64


,speaker,speakerGroup,speech
1,國場幸之助,自由民主党・無所属の会,○國場委員長 これより会議を開きます。\r\n 国際情勢に関する件について調査を進め
2,國場幸之助,自由民主党・無所属の会,○國場委員長 御異議なしと認めます。よって、そのように決しました。\r\n ―
3,國場幸之助,自由民主党・無所属の会,○國場委員長 質疑の申出がありますので、順次これを許します。石橋林太郎君。
4,石橋林太郎,自由民主党・無所属の会,○石橋委員 皆さん、おはようございます。自民党の石橋林太郎です。\r\n 茂木大臣に
6,石橋林太郎,自由民主党・無所属の会,○石橋委員 御答弁ありがとうございました。\r\n 今御答弁をいただきましたとおり、
8,石橋林太郎,自由民主党・無所属の会,○石橋委員 ありがとうございました。\r\n そのときの事態に対応して、時の政権が命
10,石橋林太郎,自由民主党・無所属の会,○石橋委員 ありがとうございます。\r\n 今、所有権又は管理権というものが、非核兵
12,石橋林太郎,自由民主党・無所属の会,○石橋委員 御答弁ありがとうございました。\r\n 正しい情報を発信することに加えて
13,國場幸之助,自由民主党・無所属の会,○國場委員長 次に、末松義規君。
14,末松義規,立憲民主党・無所属,○末松委員 茂木外務大臣には初めて質問させていただきます。\r\n 私は、今思うんで


**結果**: 立憲 43件・自民 16件。ただし自民の中には、委員長の司会(「これより会議を開きます」「次に、末松義規君。」)が入っている → **委員長の司会の発言は、別に除く必要がある**(`src/preprocess/filters.py` で対応)。

## 4. どの月まで登録されているか

2026-06〜09 の件数を数える。

In [16]:
for month in ["2026-06", "2026-07", "2026-08", "2026-09"]:
    d = fetch(**{"from": f"{month}-01", "until": f"{month}-28"})
    print(month, d.get("numberOfRecords", 0))

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-06-01&until=2026-06-28
2026-06 15791
https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-07-01&until=2026-07-28
2026-07 9439
https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-08-01&until=2026-08-28
2026-08 12
https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-09-01&until=2026-09-28
{
  "numberOfRecords": 0,
  "numberOfReturn": 0,
  "startRecord": 1,
  "nextRecordPosition": null
}
2026-09 0


**結果**(2026-10-25 時点): 2026-06 は 15,791件、2026-07 は 9,439件、2026-08 は 12件、2026-09 は 0件。

最初に調べたときは、2026-06 が 14,669件、2026-07 が 9,197件だった。**同じ月の件数が後から増えている** → 会議録は、国会が開かれてからしばらく後に少しずつ登録される。
- 取得の期間は 2025-01〜2026-07 にした(2026-08 以降は、発言がほとんど無い)
- 手元のデータ(`data/raw/kokkai/`)は最初に取得した時点のもので、2026-06・07 の発言が少し欠けている(README の「限界」に書く)

最後のセルは、受け取った 100件のサンプルを保存しているだけ。

In [17]:
with open(RAW / "sample_2025-11.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, ensure_ascii=False, indent=1)

## まとめ

- API は発言1件ごとに、発言者・会派・役職・院・会議名・本文などを返す。発言の順番 0 は会議録の見出しなので除く
- **会派の名前は、同じ議員でも時期によって変わる**(永井学議員の例)→ 会派から党を決める対応表を作る
- 役職の欄で政府側の答弁を、委員長の司会は発言の中身で、それぞれ除く(`src/preprocess/`)
- 取得の期間は 2025-01〜2026-07(`scripts/01_fetch_kokkai.py`)
- **注意**: 会議録は後から登録されるので、取得した日によって件数が変わる。また、問い合わせには名乗り(User-Agent)が必要になった(取得し直すときは `scripts/01_fetch_kokkai.py` にも同じ名乗りを入れる)